# Running inference jobs using PyTorch on the OSPool

## Introduction

In this tutorial, we will submit multiple jobs using PyTorch to the OSPool to classify pictures as cats or dogs, using the model from the [training step](../train/train.ipynb). This type of computational work is often called *inference*.

## Our code

Let's take a look at our wrapper script, infer.sh.

In [ ]:
cat infer.sh

<br>

**This script:**
1. Unzips our inference data into a directory called `data`.
2. Runs our Python script on the data using the specified model.

<p style="text-align: center;"><img src="img/infer.png" width="600px"></p>

**What if we have a lot of data?** Instead of submitting just one job, let's consider submitting multiple jobs!

<p style="text-align: center;"><img src="img/multijob.png" width="400px"></p>

## The HTCondor submit file

The HTCondor submit file, `infer.sub`, describes our training task, its inputs/outputs, and the resources we need for to run the task.

Let's take a look at the contents of `infer.sub` for this task:

In [ ]:
cat infer.sub

<br>

Many of the elements of the submit file are the same or similar to our training job. In contrast to the submit file for the training step, **we have a slightly different queue statement**:

`queue zipfile from data_list.txt`

When HTCondor sees this statement, it will loop over the values in `data_list.txt` and assign its values to `$(zipfile)` for each job. Let's double-check what `data_list.txt` contains:

In [ ]:
cat data_list.txt

<br>

`data_list.txt` contains the names of the zip files containing our images in our `data/` directory. Since there are 10 lines in this file, there will be 10 jobs submitted from this one HTCondor submit file, each working with a different zip file.

## Submit the inference jobs

To submit our training job, we use the `condor_submit` command:

In [ ]:
condor_submit infer.sub

<br>
If all goes correctly, you should see a message, like:

```
Submitting job(s)..........
10 job(s) submitted to cluster 14915971.
```

Though your cluster/job ID will be different and unique per submission.

## Monitor your job

You can then check your jobs' statuses with `condor_q`:

In [ ]:
condor_q

Or show a real-time view using the `condor_watch_q` command. This utility tracks your log file and prints a color-coded summary of your jobs every second.

> ⚠️ **Run `condor_watch_q` in the terminal**
>
> `condor_watch_q` doesn't render very well in the Jupyter notebook, so open a new Terminal tab and run it there.

Curious to know where your jobs are running? We can use the `condor_q` command again with an additional flag:

In [ ]:
condor_q -af RemoteHost

*The inference jobs are relatively fast, and should complete within minutes after starting.*

## Check the outputs of your jobs

When your jobs are complete, it is good practice to check your outputs to see that you get the expected results. Let's list information about the log, standard output, and standard error files.

In [ ]:
ls -lh logs/

<br>

If the standard error files (`.err`) have 0 bytes in them, then you *likely* don't have any errors with your job. We can also look at the standard output files (`.out`) to check what messages were printed by our job. For brevity, we'll just `tail` the end of the output files.

In [ ]:
tail logs/*.out

<br>

Lastly, we should have multiple `dog_probs_images_*.csv` files, which are the output files we want! We can check some of their outputs to make sure they contain appropriate data:

In [ ]:
head output/dog_probs_images_*csv

<br>

🌟 **Congratulations!** You've run multiple inference jobs using the GPUs available on the Open Science Pool!

## Bonus: Checking the GPUs in the OSPool

You may be curious to know what types of GPUs your jobs landed on! You can find this information in the `.log` file.

### Check the log file

You can open one of the `.log` files and search for `DeviceName`.

In [ ]:
grep DeviceName logs/*.log | cut -d '"' -f 4

## Troubleshooting

### "My job is held. What do I do?"

When HTCondor detects an issue with your job and can't proceed any further, it puts your job into a *hold* or *held* state, marked by an `H` when running `condor_q` or red `!` symbols in `condor_watch_q`.

When this happens, we recommend starting the troubleshooting process with `condor_q -hold` to get more information:

For example:

```
[user.name@ap40]$ condor_q -hold
14915682.423   user.name  7/23 18:11   47/0   The job exceeded allowed execute duration of 20:00:00
```

The message will usually hint at what you may need to do next, whether it's to fix a typo or adjust your resource requests. If you're ever unsure what to do with a specific hold message, you can always talk to an OSG Facilitator at [support@osg-htc.org](mailto:support@osg-htc.org).

### "My job ran successfully, but I don't see expected outputs. What went wrong?"

In this case, HTCondor detected that your job ran "successfully", meaning there were no fatal errors, but your scripts/code itself may not have ran successfully. When this happens, the best place to start is to look at the standard error and output files that HTCondor sends back with your jobs. These files will contain critical information in debugging your job.